# RetainIQ — Phase 2.4: Cleaning Validation & Export

## Objective

Run the final end-to-end validation suite and export the reproducible `telco_clean.csv`.

This notebook is the **Phase 2 quality gate**. If any assertion fails, the cleaned dataset should
not be approved for downstream SQL/modeling work.

## 1. Load Raw Source

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco_raw.csv")

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")

Loaded raw dataset: 7,043 rows × 50 columns


## 2. Reproduce the Cleaning Pipeline From Raw Data

In [2]:
df_clean = df_raw.copy()

# Text normalization
for c in df_clean.select_dtypes(include="object").columns:
    df_clean[c] = df_clean[c].str.strip()

# Semantic missing-value normalization
df_clean["Offer"] = df_clean["Offer"].fillna("No Offer")
df_clean["Internet Type"] = df_clean["Internet Type"].fillna("No Internet Service")

# Analytical flag
df_clean["is_new_customer"] = df_clean["Customer Status"].eq("Joined")

print(f"Clean dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]:,} columns")

Clean dataset: 7,043 rows × 51 columns


## 3. Raw vs. Clean Structural Comparison

In [3]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique Customer IDs",
        "Duplicate Customer IDs",
        "Missing Customer IDs"
    ],
    "Raw": [
        len(df_raw),
        len(df_raw.columns),
        df_raw["Customer ID"].nunique(),
        int(df_raw["Customer ID"].duplicated().sum()),
        int(df_raw["Customer ID"].isna().sum())
    ],
    "Clean": [
        len(df_clean),
        len(df_clean.columns),
        df_clean["Customer ID"].nunique(),
        int(df_clean["Customer ID"].duplicated().sum()),
        int(df_clean["Customer ID"].isna().sum())
    ]
})
comparison

,Metric,Raw,Clean
0,Rows,7043,7043
1,Columns,50,51
2,Unique Customer IDs,7043,7043
3,Duplicate Customer IDs,0,0
4,Missing Customer IDs,0,0


## 4. Post-Cleaning Missingness

In [4]:
missing_after = df_clean.isna().sum().rename("null_count").to_frame()
missing_after["null_pct"] = (missing_after["null_count"] / len(df_clean) * 100).round(2)

missing_after[missing_after["null_count"] > 0].sort_values("null_count", ascending=False)

,null_count,null_pct
Churn Category,5174,73.46
Churn Reason,5174,73.46


In [5]:
assert df_clean["Offer"].isna().sum() == 0
assert df_clean["Internet Type"].isna().sum() == 0
assert df_clean["Churn Category"].isna().sum() == df_raw["Churn Category"].isna().sum()
assert df_clean["Churn Reason"].isna().sum() == df_raw["Churn Reason"].isna().sum()

print("PASS — Missing-value contract satisfied.")

PASS — Missing-value contract satisfied.


## 5. Text Standardization Gate

In [6]:
string_cols = df_clean.select_dtypes(include="object").columns

remaining_whitespace = {
    c: int((df_clean[c].notna() & (df_clean[c] != df_clean[c].str.strip())).sum())
    for c in string_cols
}

remaining_whitespace = pd.Series(remaining_whitespace).sort_values(ascending=False)
remaining_whitespace[remaining_whitespace > 0]

Series([], dtype: int64)

In [7]:
assert int(remaining_whitespace.sum()) == 0
print("PASS — String standardization complete.")

PASS — String standardization complete.


## 6. Customer Identity & Grain Gate

In [8]:
assert len(df_clean) == len(df_raw)
assert df_clean["Customer ID"].notna().all()
assert df_clean["Customer ID"].is_unique
assert df_clean["Customer ID"].equals(df_raw["Customer ID"])

print("PASS — One-row-per-customer grain preserved.")

PASS — One-row-per-customer grain preserved.


## 7. Approved-Change Gate

In [9]:
approved_modified = {"Offer", "Internet Type"}
added_columns = sorted(set(df_clean.columns) - set(df_raw.columns))
removed_columns = sorted(set(df_raw.columns) - set(df_clean.columns))

unchanged_cols = [
    c for c in df_raw.columns
    if c not in approved_modified
]

unexpected_changes = [
    c for c in unchanged_cols
    if not df_clean[c].equals(df_raw[c])
]

pd.DataFrame({
    "Metric": [
        "Added columns",
        "Removed columns",
        "Unexpectedly changed original columns"
    ],
    "Value": [
        added_columns,
        removed_columns,
        unexpected_changes
    ]
})

,Metric,Value
0,Added columns,[is_new_customer]
1,Removed columns,[]
2,Unexpectedly changed original columns,[]


In [10]:
assert added_columns == ["is_new_customer"]
assert removed_columns == []
assert unexpected_changes == []

print("PASS — No unapproved original-column changes detected.")

PASS — No unapproved original-column changes detected.


## 8. Financial Integrity Gate

In [11]:
financial_cols = [
    "Monthly Charge",
    "Total Charges",
    "Total Refunds",
    "Total Extra Data Charges",
    "Total Long Distance Charges",
    "Total Revenue",
    "CLTV"
]

for c in financial_cols:
    assert df_clean[c].equals(df_raw[c])

print("PASS — All audited financial measures are unchanged.")

PASS — All audited financial measures are unchanged.


## 9. Transformation Summary

In [12]:
transformation_summary = pd.DataFrame([
    ["Offer", int(df_raw["Offer"].isna().sum()), int(df_clean["Offer"].isna().sum()), 'Null → "No Offer"'],
    ["Internet Type", int(df_raw["Internet Type"].isna().sum()), int(df_clean["Internet Type"].isna().sum()), 'Null → "No Internet Service"'],
    ["Churn Category", int(df_raw["Churn Category"].isna().sum()), int(df_clean["Churn Category"].isna().sum()), "Structural nulls preserved"],
    ["Churn Reason", int(df_raw["Churn Reason"].isna().sum()), int(df_clean["Churn Reason"].isna().sum()), "Structural nulls preserved"],
    ["is_new_customer", "Not present", int(df_clean["is_new_customer"].sum()), 'Created from Customer Status == "Joined"']
], columns=["Field", "Before", "After", "Decision"])

transformation_summary

,Field,Before,After,Decision
0,Offer,3877,0,"Null → ""No Offer"""
1,Internet Type,1526,0,"Null → ""No Internet Service"""
2,Churn Category,5174,5174,Structural nulls preserved
3,Churn Reason,5174,5174,Structural nulls preserved
4,is_new_customer,Not present,454,"Created from Customer Status == ""Joined"""


## 10. Export Clean Dataset

In [13]:
OUTPUT_PATH = Path("../data/telco_clean.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Exported: {OUTPUT_PATH.resolve()}")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]:,} columns")

Exported: C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\RetainIQ_phase_02_data_cleaning_full\data\telco_clean.csv
Shape: 7,043 rows × 51 columns


## 11. Final Quality Gate

In [14]:
assert df_clean.shape == (7043, 51)
assert df_clean["Customer ID"].notna().all()
assert df_clean["Customer ID"].is_unique
assert df_clean["Customer ID"].equals(df_raw["Customer ID"])

assert df_clean["Offer"].isna().sum() == 0
assert df_clean["Internet Type"].isna().sum() == 0

assert df_clean["Churn Category"].isna().sum() == df_raw["Churn Category"].isna().sum()
assert df_clean["Churn Reason"].isna().sum() == df_raw["Churn Reason"].isna().sum()

assert df_clean["is_new_customer"].dtype == bool
assert df_clean["is_new_customer"].equals(df_clean["Customer Status"].eq("Joined"))

for c in financial_cols:
    assert df_clean[c].equals(df_raw[c])

print("=" * 55)
print("PHASE 2 QUALITY GATE: PASSED")
print("Clean dataset approved for downstream work.")
print("=" * 55)

PHASE 2 QUALITY GATE: PASSED
Clean dataset approved for downstream work.


# Phase 2 Final Conclusion

The raw dataset has been transformed into a separate, reproducible analytical dataset.

### Implemented

- `Offer` nulls → **`No Offer`**
- `Internet Type` nulls → **`No Internet Service`**
- `Churn Category` nulls preserved
- `Churn Reason` nulls preserved
- Leading/trailing string whitespace stripped defensively
- `is_new_customer` created from `Customer Status == "Joined"`

### Validated

- **7,043 rows preserved**
- **51 columns** in the clean dataset
- `Customer ID` remains unique and complete
- Financial measures remain unchanged
- No unapproved original-column changes detected

### Phase 2 Status: PASSED

The dataset is ready for the next layer of RetainIQ: **Phase 3 — SQL / Star Schema Design**.